In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import Util
from steps.remi_controls import _calculate_pums_rates

util = Util('configs_pypyr')

In [4]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
tables

['/county_controls',
 '/industry_crosswalk',
 '/occupation_crosswalk',
 '/pums_households_prepared',
 '/pums_person_prepared',
 '/region_summed',
 '/region_summed_2050',
 '/region_summed_2060',
 '/regional_controls',
 '/remi',
 '/remi_ind',
 '/seed_households',
 '/seed_persons',
 '/seed_persons/meta/values_block_0/meta',
 '/seed_households/meta/values_block_0/meta',
 '/pums_person_prepared/meta/values_block_0/meta',
 '/pums_households_prepared/meta/values_block_0/meta']

In [2]:
df = util.get_table('seed_persons')

In [7]:
df.loc[df.ESR==3][['ESR','SOCP']]

,ESR,SOCP
1793,3.0,434071
1823,3.0,359011
1854,3.0,515112
1884,3.0,414010
1938,3.0,999920
...,...,...
202760,3.0,435061
202804,3.0,493023
202840,3.0,119151
202871,3.0,472031


In [36]:
hh = util.get_table('pums_households_prepared').query('gq==0').groupby(['county_id','age_head_group'])['WGTP'].sum().reset_index()

In [37]:
p = util.get_table('pums_person_prepared').groupby(['county_id','age_group','gq'])['PWGTP'].sum().reset_index().pivot_table(index=['county_id','age_group'], columns='gq', values='PWGTP', fill_value=0).reset_index()

In [38]:
p.merge(hh, left_on=['county_id','age_group'], right_on=['county_id','age_head_group'], how='left').to_csv('data/pums_population_by_age_group.csv', index=False)

In [24]:
remi = util.get_table('regional_controls')

In [25]:
forecast_year = util.get_setting("forecast_year")
category_col = "category" if "category" in remi.columns else "Category"
age_col = category_col
year_col = forecast_year
remi_age = remi.loc[
    remi[age_col].astype(str).str.contains("ages_", na=False),
    ["county_id", age_col, year_col],
].copy()

In [27]:
remi_age.to_csv('data/remi_population_by_age_group.csv', index=False)

,variable,2060
0,total_pop,5968250
1,hhpop,5844314
2,hh,2449926
3,gq,123937
0,military_employment,45070
1,non_military_employment,3982484
2,employment,4027553


In [101]:
remi_ind['industry'] = np.where(remi_ind['industry'] == '99', '98', remi_ind['industry'])

In [104]:
remi_ind = remi_ind.groupby(['county_id','industry']).agg({'employment':'sum'}).reset_index()

In [105]:
remi_ind

,county_id,industry,employment
0,53033,11,5988.153404
1,53033,21,1475.306935
2,53033,22,2617.059710
3,53033,23,146823.589708
4,53033,3133,120954.768792
...,...,...,...
75,53061,62,66502.281619
76,53061,71,11932.880735
77,53061,72,38599.246087
78,53061,81,30915.931721


In [2]:
counties = util.settings['counties']
counties = [int('53'+c) for c in counties]

def filter_lodes(df):
    geocode_col = [c for c in df.columns if c.startswith(('w_', 'h_')) and 'geocode' in c][0]
    df = df.rename(columns={geocode_col: 'geocode'})
    df['county_id'] = df['geocode'].astype(str).str[:5].astype(int)
    df = df.query('county_id.isin(@counties)')
    df = df.groupby('county_id').sum()
    df = df[[c for c in df.columns if c.startswith('CNS')]]
    return df

In [3]:
all_wac = filter_lodes(pd.read_csv('https://lehd.ces.census.gov/data/lodes/LODES8/wa/wac/wa_wac_S000_JT00_2023.csv.gz', compression='gzip'))

In [7]:
all_wac.sum(axis=1).sum()

2200704